# HW04 - 序列模型、循环神经网络、嵌入向量与注意力机制

- 姓名：付航
- 学号：20234080304


## 2 序列模型

### 2.1 理论计算题

给定字符序列：

$$
\text{``ababc''}
$$

一阶马尔可夫模型只考虑前一个字符：

$$
p(x_t \mid x_{t-1})
$$

词汇表为：

$$
V=\{a,b,c\}, \quad |V|=3
$$

序列中的相邻转移为：

$$
a \to b,\quad b \to a,\quad a \to b,\quad b \to c
$$

因此，以 `b` 为前一个字符时：

$$
C(b \to a)=1,\quad C(b \to b)=0,\quad C(b \to c)=1
$$

并且：

$$
C(b \to *)=2
$$

使用拉普拉斯平滑（加 1 平滑）：

$$
p(x \mid b)=\frac{C(b\to x)+1}{C(b\to *)+|V|}
$$

所以：

$$
p(a \mid b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4
$$

$$
p(c \mid b)=\frac{1+1}{2+3}=\frac{2}{5}=0.4
$$

**答案：**

$$
p('a' \mid 'b')=0.4, \qquad p('c' \mid 'b')=0.4
$$

### 2.2 编程题

任务：实现 `preprocess_text(text, n)`。

步骤：

1. 转小写；
2. 去除标点，只保留英文字母和空格；
3. 按空格分词；
4. 按词频从高到低构建词汇表，分配从 0 开始的整数 ID；
5. 使用滑动窗口生成长度为 `n` 的特征序列，并生成对应的下一个词标签。

这里按照题目示例保留最后一个无后续词窗口，其标签设为 `None`。

In [1]:
import re
from collections import Counter

def preprocess_text(text, n):
    """
    文本预处理并生成自回归语言模型训练样本。

    参数：
    text: str，原始文本
    n: int，滑动窗口长度

    返回：
    vocab: dict，词到整数 ID 的映射
    (features, labels): tuple，特征序列列表与标签列表
    """
    if n <= 0:
        raise ValueError("n 必须是正整数")

    # 1. 转小写，去除标点符号：只保留英文字母和空格
    text = text.lower()
    text = re.sub(r"[^a-z\s]", " ", text)

    # 2. 按空格分词，并去除多余空字符串
    tokens = text.split()

    # 3. 构建词汇表：按词频降序；词频相同则按首次出现顺序排序，保证结果稳定
    counts = Counter(tokens)
    first_pos = {}
    for i, word in enumerate(tokens):
        if word not in first_pos:
            first_pos[word] = i

    sorted_words = sorted(counts.keys(), key=lambda w: (-counts[w], first_pos[w]))
    vocab = {word: idx for idx, word in enumerate(sorted_words)}

    # 4. 滑动窗口生成特征和标签
    features = []
    labels = []
    for i in range(0, len(tokens) - n + 1):
        feature = tokens[i:i+n]
        label = tokens[i+n] if i + n < len(tokens) else None
        features.append(feature)
        labels.append(label)

    return vocab, (features, labels)

# 示例 1：题目中的例子
text1 = "The time machine"
vocab1, (features1, labels1) = preprocess_text(text1, n=2)
print("示例文本：", text1)
print("词汇表：", vocab1)
print("特征：", features1)
print("标签：", labels1)

print("-" * 60)

# 示例 2：包含大小写和标点的文本
text2 = "The Time machine, the time traveler!"
vocab2, (features2, labels2) = preprocess_text(text2, n=3)
print("示例文本：", text2)
print("词汇表：", vocab2)
print("特征：", features2)
print("标签：", labels2)

示例文本： The time machine
词汇表： {'the': 0, 'time': 1, 'machine': 2}
特征： [['the', 'time'], ['time', 'machine']]
标签： ['machine', None]
------------------------------------------------------------
示例文本： The Time machine, the time traveler!
词汇表： {'the': 0, 'time': 1, 'machine': 2, 'traveler': 3}
特征： [['the', 'time', 'machine'], ['time', 'machine', 'the'], ['machine', 'the', 'time'], ['the', 'time', 'traveler']]
标签： ['the', 'time', 'traveler', None]


## 3 循环神经网络

### 3.1 理论计算题

线性 RNN 定义为：

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t
$$

$$
o_t=W_{oh}h_t
$$

平方损失为：

$$
L=\frac{1}{2}\sum_{t=1}^{T}(o_t-y_t)^2
$$

记输出层误差为：

$$
e_t=\frac{\partial L}{\partial o_t}=o_t-y_t
$$

由于 $h_t$ 不仅影响当前输出 $o_t$，也会通过递推影响后续隐藏状态，所以需要通过时间反向传播（BPTT）。定义：

$$
\delta_t=\frac{\partial L}{\partial h_t}
$$

则有递推关系：

$$
\delta_t=W_{oh}^{T}e_t+W_{hh}^{T}\delta_{t+1}, \qquad \delta_{T+1}=0
$$

展开得到：

$$
\delta_t=\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}e_k
$$

对 $W_{hh}$ 求梯度时，当前时间步的直接关系为：

$$
h_t=W_{hh}h_{t-1}+W_{hx}x_t
$$

所以：

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}\delta_t h_{t-1}^{T}
$$

将 $\delta_t$ 的展开式代入：

$$
\frac{\partial L}{\partial W_{hh}}
=\sum_{t=1}^{T}
\left[
\sum_{k=t}^{T}(W_{hh}^{T})^{k-t}W_{oh}^{T}(o_k-y_k)
\right]h_{t-1}^{T}
$$

这就是通过时间反向传播展开到所有时间步后的梯度表达式。

**梯度消失或爆炸条件：**

梯度中包含矩阵幂 $(W_{hh}^{T})^{k-t}$。当时间跨度变长时：

- 如果 $W_{hh}$ 的谱半径 $\rho(W_{hh})<1$，矩阵幂趋近于 0，容易出现梯度消失；
- 如果 $\rho(W_{hh})>1$，矩阵幂可能快速增大，容易出现梯度爆炸；
- 如果 $\rho(W_{hh})\approx 1$，梯度更容易保持稳定，但仍会受到输入、输出误差和矩阵方向性的影响。

对于带非线性激活函数的 RNN，还需要额外考虑激活函数导数的连乘项。

### 3.2 编程题

实现一个简单 RNN 单元的前向传播和单步反向传播。

前向传播：

$$
a_t=x_tW_{hx}+h_{t-1}W_{hh}+b_h
$$

$$
h_t=\tanh(a_t)
$$

反向传播：

$$
da_t=dh_{next}\odot(1-h_t^2)
$$

$$
dx_t=da_tW_{hx}^{T}
$$

$$
dh_{prev}=da_tW_{hh}^{T}
$$

$$
dW_{hx}=x_t^Tda_t
$$

$$
dW_{hh}=h_{prev}^Tda_t
$$

$$
db_h=\sum_{batch} da_t
$$

In [2]:
import numpy as np

def rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h):
    """
    单个 RNN 单元前向传播。

    x_t:    (batch_size, input_size)
    h_prev: (batch_size, hidden_size)
    W_hx:   (input_size, hidden_size)
    W_hh:   (hidden_size, hidden_size)
    b_h:    (hidden_size,)
    """
    a_t = x_t @ W_hx + h_prev @ W_hh + b_h
    h_t = np.tanh(a_t)
    cache = (x_t, h_prev, W_hx, W_hh, b_h, a_t, h_t)
    return h_t, cache


def rnn_cell_backward(dh_next, cache):
    """
    单个 RNN 单元单步反向传播。

    dh_next: 损失对当前隐藏状态 h_t 的上游梯度
    """
    x_t, h_prev, W_hx, W_hh, b_h, a_t, h_t = cache

    # tanh'(a) = 1 - tanh(a)^2
    da_t = dh_next * (1 - h_t ** 2)

    dx_t = da_t @ W_hx.T
    dh_prev = da_t @ W_hh.T
    dW_hx = x_t.T @ da_t
    dW_hh = h_prev.T @ da_t
    db_h = da_t.sum(axis=0)

    return dx_t, dh_prev, dW_hx, dW_hh, db_h

# 测试代码
np.random.seed(42)
batch_size = 2
input_size = 3
hidden_size = 4

x_t = np.random.randn(batch_size, input_size)
h_prev = np.random.randn(batch_size, hidden_size)
W_hx = np.random.randn(input_size, hidden_size)
W_hh = np.random.randn(hidden_size, hidden_size)
b_h = np.random.randn(hidden_size)
dh_next = np.random.randn(batch_size, hidden_size)

h_t, cache = rnn_cell_forward(x_t, h_prev, W_hx, W_hh, b_h)
dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell_backward(dh_next, cache)

print("h_t shape:", h_t.shape)
print("dx_t shape:", dx_t.shape)
print("dh_prev shape:", dh_prev.shape)
print("dW_hx shape:", dW_hx.shape)
print("dW_hh shape:", dW_hh.shape)
print("db_h shape:", db_h.shape)

print("\nh_t:\n", np.round(h_t, 4))
print("\ndb_h:\n", np.round(db_h, 4))

h_t shape: (2, 4)
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (4,)

h_t:
 [[-0.9995  0.8825 -0.9966 -0.617 ]
 [ 0.7649 -0.9758 -0.9996 -0.3702]]

db_h:
 [ 0.1341  0.2155  0.0018 -0.5641]


## 4 高级循环神经网络

### 4.1 理论计算题

设深度双向 RNN 有：

- 层数：$L$
- 每个方向的隐藏单元数：$H$
- 输入维度：$D$
- 输出维度：$O$

这里按普通 RNN 计算，每个方向每层包括：

1. 输入到隐藏层权重；
2. 隐藏到隐藏层权重；
3. 隐藏层偏置。

#### 第 1 层参数量

第 1 层每个方向的输入维度为 $D$，因此每个方向参数量为：

$$
D H + H^2 + H
$$

双向 RNN 有前向和后向两个方向，所以第 1 层参数量为：

$$
2(DH+H^2+H)
$$

#### 第 2 层到第 L 层参数量

从第 2 层开始，每一层的输入是上一层前向和后向隐藏状态的拼接，因此输入维度为 $2H$。

每个方向参数量为：

$$
2H\cdot H + H^2 + H=3H^2+H
$$

双向后每层参数量为：

$$
2(3H^2+H)=6H^2+2H
$$

因此第 2 层到第 $L$ 层总参数量为：

$$
(L-1)(6H^2+2H)
$$

#### 输出层参数量

最后输出层输入为前向和后向最终隐藏状态的拼接，维度为 $2H$，输出维度为 $O$：

$$
2HO+O
$$

#### 总参数量

$$
\boxed{
N=2(DH+H^2+H)+(L-1)(6H^2+2H)+2HO+O
}
$$

说明：这里假设每个 RNN 层每个方向只有一个隐藏层偏置。如果使用 PyTorch 的 `nn.RNN`，默认每层每个方向有 `bias_ih` 和 `bias_hh` 两个偏置，则偏置项需要相应增加。

### 4.2 编程题

实现双向 RNN 编码器。输入形状为：

$$
(seq\_len, batch, input\_dim)
$$

返回：

1. 每个时间步的前向和后向隐藏状态拼接，形状为：

$$
(seq\_len, batch, 2\times hidden\_dim)
$$

2. 最后一层前向最终状态与后向最终状态拼接后的序列表示，形状为：

$$
(batch, 2\times hidden\_dim)
$$

In [3]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim, num_layers=1):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            bidirectional=True,
            batch_first=False,
            nonlinearity="tanh"
        )
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers

    def forward(self, X):
        """
        X: (seq_len, batch, input_dim)
        outputs: (seq_len, batch, 2 * hidden_dim)
        final_repr: (batch, 2 * hidden_dim)
        """
        outputs, h_n = self.rnn(X)

        # h_n shape: (num_layers * 2, batch, hidden_dim)
        # 最后一层前向隐藏状态为 h_n[-2]
        # 最后一层后向隐藏状态为 h_n[-1]
        final_forward = h_n[-2]
        final_backward = h_n[-1]
        final_repr = torch.cat([final_forward, final_backward], dim=1)

        return outputs, final_repr

# 测试代码
torch.manual_seed(42)
seq_len = 5
batch = 3
input_dim = 4
hidden_dim = 6
num_layers = 2

X = torch.randn(seq_len, batch, input_dim)
encoder = BiRNNEncoder(input_dim=input_dim, hidden_dim=hidden_dim, num_layers=num_layers)
outputs, final_repr = encoder(X)

print("X shape:", tuple(X.shape))
print("outputs shape:", tuple(outputs.shape))
print("final_repr shape:", tuple(final_repr.shape))
print("\nfinal_repr 前两行：")
print(final_repr[:2].detach())

X shape: (5, 3, 4)
outputs shape: (5, 3, 12)
final_repr shape: (3, 12)

final_repr 前两行：
tensor([[-0.5491, -0.1163, -0.0863,  0.5260, -0.4603,  0.1116, -0.7066, -0.6346,
         -0.3874,  0.7200, -0.2890, -0.7181],
        [-0.3307, -0.7079,  0.3569,  0.1531,  0.1871, -0.4676, -0.4457, -0.5635,
         -0.1789,  0.7396, -0.5527, -0.8809]])


## 5 嵌入向量

### 5.1 理论计算题

在 Skip-gram 模型中，给定中心词 $w_c$ 和上下文词 $w_o$。

设中心词向量为：

$$
v_c
$$

正样本上下文词向量为：

$$
u_o
$$

负样本为：

$$
w_{n_1},w_{n_2},\dots,w_{n_K}
$$

对应的负样本词向量为：

$$
u_{n_1},u_{n_2},\dots,u_{n_K}
$$

对于正样本 $(w_c,w_o)$，希望内积 $u_o^T v_c$ 尽可能大；对于负样本 $(w_c,w_{n_k})$，希望内积 $u_{n_k}^T v_c$ 尽可能小。

因此，负采样 Skip-gram 的对数似然目标为：

$$
\log \sigma(u_o^Tv_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^Tv_c)
$$

其中：

$$
\sigma(x)=\frac{1}{1+e^{-x}}
$$

如果写成需要最小化的损失函数，则为负对数似然：

$$
\boxed{
\mathcal{L}
=-\left[
\log \sigma(u_o^Tv_c)+\sum_{k=1}^{K}\log \sigma(-u_{n_k}^Tv_c)
\right]
}
$$

负样本通常从噪声分布 $P_n(w)$ 中采样。Word2Vec 中常用的噪声分布是词频的 $3/4$ 次幂归一化：

$$
P_n(w)=\frac{f(w)^{3/4}}{\sum_{w'\in V}f(w')^{3/4}}
$$

其中 $f(w)$ 表示词 $w$ 在语料中的出现频率。实际采样时，从该分布中独立采样 $K$ 个负样本，通常避免把真实上下文词作为负样本。

### 5.2 编程题

实现 CBOW 模型的前向传播和完整 softmax 交叉熵损失。

对于一个样本，若上下文词索引为：

$$
w_1,w_2,\dots,w_C
$$

嵌入矩阵为：

$$
W\in \mathbb{R}^{V\times d}
$$

输出权重矩阵为：

$$
W_{out}\in \mathbb{R}^{d\times V}
$$

上下文词向量平均得到隐藏层：

$$
h=\frac{1}{C}\sum_{i=1}^{C}W[w_i]
$$

输出 logits：

$$
z=hW_{out}
$$

softmax 概率：

$$
p_j=\frac{e^{z_j}}{\sum_{k=1}^{V}e^{z_k}}
$$

目标中心词索引为 $y$，交叉熵损失为：

$$
\mathcal{L}=-\log p_y
$$

In [4]:
import numpy as np


def softmax(logits):
    """稳定版 softmax。"""
    shifted = logits - np.max(logits, axis=1, keepdims=True)
    exp_logits = np.exp(shifted)
    return exp_logits / np.sum(exp_logits, axis=1, keepdims=True)


def cbow_forward_loss(context_indices, target_indices, W, W_out):
    """
    CBOW 前向传播和完整 softmax 交叉熵损失。

    context_indices: (batch_size, context_size)，上下文词索引
    target_indices:  (batch_size,)，中心词索引
    W:               (V, d)，输入嵌入矩阵
    W_out:           (d, V)，输出权重矩阵
    """
    context_vectors = W[context_indices]          # (batch_size, context_size, d)
    hidden = context_vectors.mean(axis=1)         # (batch_size, d)
    logits = hidden @ W_out                       # (batch_size, V)
    probs = softmax(logits)                       # (batch_size, V)

    batch_size = context_indices.shape[0]
    loss = -np.mean(np.log(probs[np.arange(batch_size), target_indices] + 1e-12))
    return loss, probs, hidden

# 测试代码
np.random.seed(42)
V = 8
d = 5
batch_size = 3
context_size = 4

W = np.random.randn(V, d) * 0.1
W_out = np.random.randn(d, V) * 0.1
context_indices = np.array([
    [0, 1, 2, 3],
    [1, 2, 3, 4],
    [2, 3, 4, 5]
])
target_indices = np.array([4, 5, 6])

loss, probs, hidden = cbow_forward_loss(context_indices, target_indices, W, W_out)

print("context_indices shape:", context_indices.shape)
print("hidden shape:", hidden.shape)
print("probs shape:", probs.shape)
print("loss:", round(float(loss), 6))
print("\n每个样本预测概率之和：", np.round(probs.sum(axis=1), 6))
print("第一个样本的概率分布：\n", np.round(probs[0], 4))

context_indices shape: (3, 4)
hidden shape: (3, 5)
probs shape: (3, 8)
loss: 2.078015

每个样本预测概率之和： [1. 1. 1.]
第一个样本的概率分布：
 [0.1241 0.1229 0.1279 0.1246 0.125  0.1259 0.1244 0.1251]


## 6 注意力机制

### 6.1 理论计算题

题目给定：

$$
Q\in \mathbb{R}^{2\times 4},\quad K\in \mathbb{R}^{3\times 4},\quad V\in \mathbb{R}^{3\times 5}
$$

缩放点积注意力为：

$$
\mathrm{Attention}(Q,K,V)=\mathrm{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V
$$

其中：

$$
d_k=4,\quad \sqrt{d_k}=2
$$

设：

$$
Q=
\begin{bmatrix}
q_1^T\\
q_2^T
\end{bmatrix}
,
\quad
K=
\begin{bmatrix}
k_1^T\\
k_2^T\\
k_3^T
\end{bmatrix}
,
\quad
V=
\begin{bmatrix}
v_1^T\\
v_2^T\\
v_3^T
\end{bmatrix}
$$

其中 $q_i,k_j\in\mathbb{R}^4$，$v_j\in\mathbb{R}^5$。

#### 第一步：计算得分矩阵

$$
S=\frac{QK^T}{\sqrt{d_k}}=\frac{QK^T}{2}
$$

$$
S=
\frac{1}{2}
\begin{bmatrix}
q_1\cdot k_1 & q_1\cdot k_2 & q_1\cdot k_3\\
q_2\cdot k_1 & q_2\cdot k_2 & q_2\cdot k_3
\end{bmatrix}
$$

因此：

$$
S\in \mathbb{R}^{2\times 3}
$$

#### 第二步：对每一行做 softmax

设：

$$
A=\mathrm{softmax}(S)
$$

则：

$$
A_{ij}=\frac{e^{S_{ij}}}{\sum_{m=1}^{3}e^{S_{im}}}
$$

所以：

$$
A\in \mathbb{R}^{2\times 3}
$$

#### 第三步：加权求和

输出矩阵为：

$$
O=AV
$$

即：

$$
O=
\begin{bmatrix}
A_{11}v_1^T+A_{12}v_2^T+A_{13}v_3^T\\
A_{21}v_1^T+A_{22}v_2^T+A_{23}v_3^T
\end{bmatrix}
$$

因为 $A\in\mathbb{R}^{2\times3}$，$V\in\mathbb{R}^{3\times5}$，所以：

$$
\boxed{O\in\mathbb{R}^{2\times5}}
$$

这就是无掩码缩放点积注意力的完整计算过程。

### 6.2 编程题

实现多头注意力前向传播，设：

$$
num\_heads=2,\quad d\_model=4
$$

因此每个头的维度为：

$$
d_k=d_v=\frac{d_{model}}{num\_heads}=2
$$

输入：

$$
X\in\mathbb{R}^{seq\_len\times batch\times d\_model}
$$

输出形状与输入相同：

$$
(seq\_len,batch,d\_model)
$$

In [5]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model=4, num_heads=2):
        super().__init__()
        if d_model % num_heads != 0:
            raise ValueError("d_model 必须能被 num_heads 整除")

        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads

        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)

    def forward(self, X):
        """
        X: (seq_len, batch, d_model)
        return: (seq_len, batch, d_model)
        """
        seq_len, batch, d_model = X.shape

        Q = self.W_q(X)
        K = self.W_k(X)
        V = self.W_v(X)

        # reshape: (seq_len, batch, d_model)
        # -> (seq_len, batch, num_heads, d_k)
        # -> (batch, num_heads, seq_len, d_k)
        Q = Q.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        K = K.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)
        V = V.view(seq_len, batch, self.num_heads, self.d_k).permute(1, 2, 0, 3)

        # 缩放点积注意力
        scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.d_k)
        attn_weights = F.softmax(scores, dim=-1)
        head_outputs = torch.matmul(attn_weights, V)

        # 拼接所有头：
        # (batch, num_heads, seq_len, d_k)
        # -> (seq_len, batch, num_heads, d_k)
        # -> (seq_len, batch, d_model)
        concat = head_outputs.permute(2, 0, 1, 3).contiguous().view(seq_len, batch, d_model)

        output = self.W_o(concat)
        return output

# 测试代码
torch.manual_seed(42)
seq_len = 4
batch = 2
d_model = 4
num_heads = 2

X = torch.randn(seq_len, batch, d_model)
mha = MultiHeadAttention(d_model=d_model, num_heads=num_heads)
output = mha(X)

print("X shape:", tuple(X.shape))
print("output shape:", tuple(output.shape))
print("\noutput 前两个时间步：")
print(output[:2].detach())

X shape: (4, 2, 4)
output shape: (4, 2, 4)

output 前两个时间步：
tensor([[[-0.4449, -0.3245,  0.4447,  0.3020],
         [-0.3162, -0.3904,  0.8990,  0.0598]],

        [[-0.3652, -0.1852,  0.4880,  0.3777],
         [-0.3335, -0.5424,  0.9919, -0.0143]]])
